In [1]:
import genetick_AI
import pandas
import numpy as np

def f_2(x):
    return ((1 - x) ** 76)/((1 - x) ** 76 + 10 * x ** 12)

def f_3(x):
    return (1-x)*(1-f_2(x)-f_5(x))

def f_4(x):
    return f_3(1 - x)

def f_5(x):
    return f_2(1 - x)

In [2]:
n,m = 10,10
matrix, perf, diff = genetick_AI.generate_rating_matrix(n,m)
df = pandas.DataFrame(matrix)
df

,0,1,2,3,4,5,6,7,8,9
0,4.0,4.0,3.0,4.0,4.0,4.0,3.0,4.0,4.0,3.0
1,4.0,4.0,4.0,3.0,4.0,3.0,4.0,3.0,4.0,4.0
2,3.0,4.0,3.0,4.0,4.0,4.0,3.0,3.0,3.0,4.0
3,3.0,4.0,3.0,3.0,4.0,4.0,4.0,3.0,4.0,3.0
4,3.0,5.0,3.0,3.0,4.0,3.0,4.0,4.0,4.0,3.0
5,3.0,3.0,3.0,3.0,3.0,4.0,4.0,3.0,3.0,3.0
6,4.0,3.0,4.0,4.0,3.0,4.0,4.0,3.0,4.0,4.0
7,4.0,4.0,4.0,3.0,3.0,4.0,4.0,3.0,3.0,3.0
8,3.0,3.0,3.0,4.0,4.0,3.0,3.0,3.0,3.0,3.0
9,3.0,3.0,4.0,4.0,4.0,3.0,4.0,4.0,4.0,3.0


In [3]:
def calculate_f_matrix(ratings_matrix, st_it, student_alpha=2, student_beta=2,
                       item_alpha=2, item_beta=2):
    """
    Returns a matrix of f_rating contributions (without Beta priors) for observed ratings.

    Parameters
    ----------
    ratings_matrix : np.ndarray, shape (n_students, n_items)
        Observed ratings (2..5). Unobserved entries should be 0, NaN, or a sentinel.
    st_it : np.ndarray
        Concatenated vector: first n_students entries = student parameters,
        remaining = item parameters. All values in (0,1).
    student_alpha, student_beta, item_alpha, item_beta : float
        Ignored (kept only for API compatibility). Excluded from calculation.

    Returns
    -------
    f_matrix : np.ndarray, same shape as ratings_matrix
        For each observed rating, f_rating( student_i / (student_i + item_j) ).
        Unobserved cells are set to 0.0.
    """
    # Dictionary mapping rating values to their corresponding functions
    methods = {2: f_2, 3: f_3, 4: f_4, 5: f_5}   # f_2..f_5 must be defined elsewhere

    n_students, n_items = ratings_matrix.shape
    students = st_it[:n_students]
    items = st_it[n_students:]

    # Initialize matrix with zeros (unrated cells stay 0)
    f_matrix = np.zeros_like(ratings_matrix, dtype=float)

    # Fill f_rating contributions for each observed rating value
    for rating, func in methods.items():
        mask = (ratings_matrix == rating)
        if not np.any(mask):
            continue
        rows, cols = np.where(mask)
        v1_vals = students[rows]
        v2_vals = items[cols]
        ratio = v1_vals / (v1_vals + v2_vals)
        f_vals = func(ratio)
        f_matrix[rows, cols] = f_vals

    return f_matrix
a = np.concatenate((perf,diff))
prob_m = calculate_f_matrix(matrix,a)
df = pandas.DataFrame(prob_m)
df

,0,1,2,3,4,5,6,7,8,9
0,0.420816,0.756734,0.492385,0.533699,0.712005,0.492119,0.301524,0.544310,0.617383,0.415763
1,0.371106,0.717927,0.455719,0.518255,0.667544,0.559611,0.652943,0.507586,0.567192,0.532986
2,0.578009,0.757422,0.491181,0.534897,0.712991,0.493322,0.300510,0.454496,0.381480,0.585406
3,0.729530,0.615232,0.655307,0.631326,0.557800,0.330829,0.541689,0.621332,0.451539,0.582421
4,0.484228,0.967430,0.398194,0.373430,0.721186,0.413139,0.760528,0.636507,0.702867,0.326794
5,0.750632,0.448378,0.711888,0.689984,0.507473,0.275561,0.476270,0.680775,0.612204,0.644478
6,0.472257,0.143716,0.559418,0.585001,0.247021,0.544088,0.740370,0.404670,0.665253,0.633794
7,0.303024,0.652208,0.381532,0.593515,0.403324,0.367015,0.580917,0.583172,0.508764,0.543223
8,0.661934,0.312218,0.579821,0.445838,0.634750,0.594844,0.380470,0.543588,0.468553,0.503077
9,0.540481,0.191751,0.546765,0.572521,0.742986,0.468637,0.730486,0.582939,0.653758,0.378167


In [4]:
del a
res = genetick_AI.try_find_opt(matrix)

In [5]:
# print(pandas.Series(res[:n]),'\n',pandas.Series(res[n:]))
# print(pandas.Series(perf),'\n', pandas.Series(diff))
df1 = pandas.DataFrame((pandas.Series(res[:n]),pandas.Series(perf)))
df2 = pandas.DataFrame((pandas.Series(res[n:]),pandas.Series(diff)))
df1

,0,1,2,3,4,5,6,7,8,9
0,0.632296,0.620877,0.514066,0.448641,0.830836,0.308883,0.639906,0.537444,0.200409,0.562723
1,0.615635,0.499997,0.618608,0.314110,0.902516,0.241679,0.758234,0.368389,0.432748,0.720396


In [6]:
df2

,0,1,2,3,4,5,6,7,8,9
0,0.639246,0.199269,0.586292,0.443945,0.306136,0.478696,0.295163,0.651079,0.351908,0.672435
1,0.847320,0.196445,0.597164,0.537891,0.249013,0.635355,0.265762,0.515403,0.381534,0.438108


In [7]:
print(np.abs(perf-res[:n]),'\n',np.abs(diff-res[n:]))
print(max(np.abs(perf-res[:n])),max(np.abs(diff-res[n:])))

[0.01666101 0.12088085 0.10454156 0.13453089 0.07168013 0.0672035
 0.11832789 0.1690546  0.23233855 0.15767234] 
 [0.20807404 0.00282347 0.01087233 0.09394526 0.05712329 0.15665887
 0.02940085 0.13567561 0.02962549 0.23432731]
0.23233854512795965 0.2343273113116674


In [8]:
prob_m_ap = calculate_f_matrix(matrix,np.array(res))
df1 = pandas.DataFrame(prob_m_ap)
df1

,0,1,2,3,4,5,6,7,8,9
0,0.497267,0.758449,0.481124,0.587504,0.673779,0.569128,0.318249,0.492682,0.642444,0.515382
1,0.492712,0.755872,0.514325,0.416920,0.669761,0.435347,0.677784,0.511872,0.638247,0.480068
2,0.554270,0.720647,0.532819,0.536597,0.626755,0.517814,0.364746,0.558796,0.406373,0.433262
3,0.587603,0.692444,0.566502,0.497369,0.594402,0.483795,0.603171,0.592040,0.560417,0.599812
4,0.434837,0.744942,0.413718,0.348252,0.730722,0.365547,0.737801,0.560650,0.702464,0.447315
5,0.674218,0.392144,0.654947,0.589703,0.497767,0.392193,0.511356,0.678234,0.532556,0.685237
6,0.500258,0.236628,0.521862,0.590400,0.323597,0.572059,0.684341,0.504327,0.645188,0.487607
7,0.456742,0.729498,0.478265,0.452364,0.362901,0.528908,0.645495,0.547805,0.395690,0.555787
8,0.759103,0.498573,0.745057,0.311023,0.395639,0.704892,0.595600,0.760980,0.637148,0.761681
9,0.531832,0.261485,0.489744,0.558996,0.647657,0.459657,0.655941,0.463604,0.615246,0.544412


In [9]:
df

,0,1,2,3,4,5,6,7,8,9
0,0.420816,0.756734,0.492385,0.533699,0.712005,0.492119,0.301524,0.544310,0.617383,0.415763
1,0.371106,0.717927,0.455719,0.518255,0.667544,0.559611,0.652943,0.507586,0.567192,0.532986
2,0.578009,0.757422,0.491181,0.534897,0.712991,0.493322,0.300510,0.454496,0.381480,0.585406
3,0.729530,0.615232,0.655307,0.631326,0.557800,0.330829,0.541689,0.621332,0.451539,0.582421
4,0.484228,0.967430,0.398194,0.373430,0.721186,0.413139,0.760528,0.636507,0.702867,0.326794
5,0.750632,0.448378,0.711888,0.689984,0.507473,0.275561,0.476270,0.680775,0.612204,0.644478
6,0.472257,0.143716,0.559418,0.585001,0.247021,0.544088,0.740370,0.404670,0.665253,0.633794
7,0.303024,0.652208,0.381532,0.593515,0.403324,0.367015,0.580917,0.583172,0.508764,0.543223
8,0.661934,0.312218,0.579821,0.445838,0.634750,0.594844,0.380470,0.543588,0.468553,0.503077
9,0.540481,0.191751,0.546765,0.572521,0.742986,0.468637,0.730486,0.582939,0.653758,0.378167


In [10]:
print(df.prod().prod(),df1.prod().prod(),(df1/df).prod().prod(),df1.prod().prod()/df.prod().prod())

2.032744503823956e-29 5.537822759407588e-28 27.243083176414725 27.24308317641471


In [11]:
import Parser_lib

In [12]:
sm1,sm2 = Parser_lib.parse('Grades.xls',sheet_name='24-25 2к 1п')

In [13]:
res1 = genetick_AI.try_find_opt(sm1)
res2 = genetick_AI.try_find_opt(sm2)

In [14]:
n_students, n_items = sm1.shape
res1, res2 = np.array(res1), np.array(res2)

In [15]:
print(np.abs(res1[:n_students]-res2[:n_students]))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [16]:
print(res1[n_students:], res2[n_students:])

[0.43686934 0.47358308 0.63581661 0.50190306 0.70166175 0.58508481
 0.11838558 0.55647711] [0.43686934 0.47358308 0.63581661 0.50190306 0.70166175 0.58508481
 0.11838558 0.55647711 0.68591861 0.68779189 0.68478055 0.64676632]


In [17]:
df1 = pandas.DataFrame(sm1)
df1

,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.0,5.0,3.0,3.0,3.0,3.0,3.0,4.0
2,5.0,4.0,5.0,4.0,4.0,5.0,5.0,4.0
3,5.0,5.0,5.0,5.0,4.0,5.0,5.0,5.0
4,5.0,4.0,3.0,4.0,3.0,5.0,4.0,3.0
...,...,...,...,...,...,...,...,...
83,5.0,5.0,5.0,4.0,3.0,4.0,5.0,4.0
84,4.0,4.0,4.0,3.0,4.0,5.0,3.0,5.0
85,3.0,3.0,4.0,4.0,3.0,4.0,4.0,3.0
86,3.0,4.0,5.0,3.0,3.0,4.0,4.0,3.0


In [18]:
df2 = pandas.DataFrame(sm2)
df2

,0,1,2,3,4,5,6,7,8,9,10,11
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.0,5.0,3.0,4.0,5.0,3.0,4.0,3.0,3.0,3.0,3.0,4.0
2,5.0,5.0,5.0,4.0,5.0,4.0,4.0,3.0,5.0,5.0,5.0,4.0
3,4.0,5.0,5.0,5.0,5.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0
4,5.0,4.0,3.0,5.0,5.0,3.0,4.0,3.0,5.0,3.0,3.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...
83,5.0,5.0,5.0,5.0,5.0,3.0,5.0,4.0,4.0,4.0,4.0,4.0
84,4.0,4.0,4.0,4.0,3.0,4.0,3.0,4.0,5.0,4.0,3.0,4.0
85,3.0,5.0,4.0,3.0,4.0,3.0,3.0,3.0,4.0,3.0,3.0,3.0
86,3.0,4.0,5.0,3.0,5.0,3.0,3.0,3.0,4.0,3.0,4.0,3.0
